## PPK Operations Analytics Case Study

In [3]:
import sys
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl

In [4]:
offices = pd.read_csv("C:/Users/rahulg/Downloads/PPK_Ops_Analytics_Case3/PPK_Ops_Analytics_Case/02_Data/offices.csv", dtype=str)
proposals_raw = pd.read_csv("C:/Users/rahulg/Downloads/PPK_Ops_Analytics_Case3/PPK_Ops_Analytics_Case/02_Data/proposals.csv", dtype=str)
events_raw = pd.read_csv("C:/Users/rahulg/Downloads/PPK_Ops_Analytics_Case3/PPK_Ops_Analytics_Case/02_Data/status_events.csv", dtype=str, keep_default_na=False)

In [5]:
summary = pd.DataFrame({
    "DataFrame": ["offices", "proposals_raw", "events_raw"],
    "Rows": [offices.shape[0], proposals_raw.shape[0], events_raw.shape[0]],
    "Columns": [offices.shape[1], proposals_raw.shape[1], events_raw.shape[1]]
})

display(summary)

,DataFrame,Rows,Columns
0,offices,14,5
1,proposals_raw,24722,6
2,events_raw,64491,3


In [6]:
proposals_raw.head()

,proposal_id,submitted_date,office,client_name,practice,proposal_value_usd
0,0000995,24/12/2025,Chicago,CL-4322,M&A,278900.0
1,0014916,2026-03-20,Sydney,CL-6352,Performance Improvement,204000.0
2,0010088,2025-01-30,Tokyo,CL-6689,Financial Services,914100.0
3,0005300,2025-01-13,Munich,CL-8209,Customer Strategy & Marketing,341800.0
4,0001767,07/08/2025,Paris,CL-6153,Digital,206000.0


In [14]:
events_raw.head()

,proposal_id,event_date,status
0,10131,2025-01-01,Submitted
1,10461,2025-01-01,Submitted
2,10522,2025-01-01,Submitted
3,10883,2025-01-01,Submitted
4,10917,2025-01-01,Submitted


In [10]:
print("Offices:\n", proposals_raw["office"].value_counts().sort_index().to_string(), "\n")
print("Practices:\n", proposals_raw["practice"].value_counts().sort_index().to_string(), "\n")

# Replace each digit with 9 to see the *shape* of each date
print("Date formats:\n", proposals_raw["submitted_date"].str.replace(r"\d", "9", regex=True)
      .str.replace(r"[A-Za-z]{3}", "Mon", regex=True).value_counts())

Offices:
 office
BOSTON           6
Boston        1934
Boston          37
CHICAGO         13
Chicago       1931
Chicago         32
DUBAI            9
Dubai         1162
Dubai           18
LONDON           8
London        1855
London          36
MADRID           8
MUNICH           7
Madrid        1795
Madrid          25
Munich        1757
Munich          20
NEW DELHI        5
NEW YORK        15
New Delhi     1198
New Delhi       25
New York      1945
New York        31
PARIS           12
Paris         1797
Paris           28
RIYADH           5
Riyadh        1077
Riyadh          19
SAO PAULO       13
SINGAPORE       12
SYDNEY           9
Sao Paulo     1911
Sao Paulo       33
Singapore     1845
Singapore       25
Sydney        1871
Sydney          30
TOKYO           13
Tokyo         1891
Tokyo           14
boston          26
chicago         20
dubai           14
london          24
madrid          13
munich          10
new delhi       17
new york        18
paris           17
riyadh        

In [8]:
values = proposals_raw["proposal_value_usd"].astype(float)
print(values.describe().apply(lambda x: f"{x:,.1f}"))
print("\nValues under 1,000:", (values < 1_000).sum())
print("Values over 10M   :", (values > 10_000_000).sum())
print("Example small values:", values[values < 1_000].head(8).tolist())

count           24,722.0
mean           402,646.7
std          9,564,204.2
min                 30.0
25%            140,600.0
50%            244,400.0
75%            422,900.0
max      1,503,100,000.0
Name: proposal_value_usd, dtype: str

Values under 1,000: 604
Values over 10M   : 1
Example small values: [208.3, 285.4, 210.1, 227.1, 343.3, 534.6, 475.4, 207.7]


In [19]:
p, e = proposals_raw, events_raw
print("Exact duplicate proposal rows:", p.duplicated().sum())
print("Duplicate proposal_ids        :", p["proposal_id"].duplicated().sum())
print("Events whose ID matches a proposal:", e["proposal_id"].isin(p["proposal_id"]).mean().round(3))
print("Sample IDs  proposals:", p["proposal_id"].head(3).tolist(), " events:", e["proposal_id"].head(3).tolist())

non_numeric = p[~p["proposal_id"].str.fullmatch(r"\d+")]
print("\nNon-numeric IDs:", len(non_numeric))
non_numeric.head()

Exact duplicate proposal rows: 300
Duplicate proposal_ids        : 300
Events whose ID matches a proposal: 0.002
Sample IDs  proposals: ['0000995', '0014916', '0010088']  events: ['10131', '10461', '10522']

Non-numeric IDs: 50


,proposal_id,submitted_date,office,client_name,practice,proposal_value_usd
546,ZZ00038,07 May 2026,Tokyo,Test - do not use,Consumer Products,1000000.0
591,ZZ00011,2025-10-06,New Delhi,TEST CLIENT,M&A,5000000.0
928,ZZ00049,17/07/2026,Riyadh,TEST CLIENT,Industrials,1000000.0
1073,ZZ00019,06/11/2025,New Delhi,Test - do not use,Strategy,5000000.0
1831,ZZ00005,26/12/2025,London,Dummy Co,Performance Improvement,5000000.0


In [12]:
# Slash dates: is it DD/MM/YYYY or MM/DD/YYYY?
slash = p.loc[p["submitted_date"].str.contains("/"), "submitted_date"].str.split("/", expand=True).astype(int)
print("First part > 12 :", (slash[0] > 12).sum())
print("Second part > 12:", (slash[1] > 12).sum())
# If only the first part is ever > 12, the format must be DD/MM/YYYY.

First part > 12 : 3435
Second part > 12: 0


In [13]:
# Practice names over time: does a practice appear or vanish?
tmp = p.assign(practice_l=p["practice"].str.strip().str.lower(),
               year=p["submitted_date"].str.extract(r"(20\d\d)")[0])
pd.crosstab(tmp["practice_l"], tmp["year"])

year,2025,2026
practice_l,,
consumer products,1103,996
customer,0,961
customer strategy & marketing,1151,0
digital,1248,1088
energy & natural resources,895,710
financial services,1317,1170
healthcare,1192,1050
industrials,984,858
m&a,1268,1007


### Clean Proposals table

In [20]:
p = proposals_raw.drop_duplicates().copy()                                  # issue 1

p["proposal_id"] = p["proposal_id"].str.strip().str.lstrip("0")            # issue 2
is_test = p["proposal_id"].str.startswith("ZZ") | p["client_name"].str.contains("test|dummy", case=False)
test_ids = set(p.loc[is_test, "proposal_id"])
p = p[~is_test]                                                             # issue 3
print("Test records removed:", len(test_ids))

def parse_date(text):                                                       # issue 4
    text = str(text).strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d %b %Y"):
        try:
            return pd.to_datetime(text, format=fmt)
        except ValueError:
            pass
    return pd.NaT

p["submitted_date"] = p["submitted_date"].map(parse_date)
print("Dates that failed to parse:", p["submitted_date"].isna().sum())

office_lookup = {o.lower(): o for o in offices["office"]}                   # issue 5
p["office"] = p["office"].str.strip().str.lower().map(office_lookup)
print("Unmatched offices:", p["office"].isna().sum())

practice_lookup = {                                                         # issues 6 and 7
    "strategy": "Strategy",
    "technology, media & telecom": "Technology, Media & Telecom",
    "financial services": "Financial Services",
    "digital": "Digital",
    "m&a": "M&A",
    "healthcare": "Healthcare",
    "consumer products": "Consumer Products",
    "performance improvement": "Performance Improvement",
    "industrials": "Industrials",
    "energy & natural resources": "Energy & Natural Resources",
    "customer strategy & marketing": "Customer Strategy & Marketing",
    "customer": "Customer Strategy & Marketing",   # renamed in 2026 -> merged
}
p["practice"] = p["practice"].str.strip().str.lower().map(practice_lookup)
print("Unmatched practices:", p["practice"].isna().sum())

value = p["proposal_value_usd"].astype(float)                               # issues 8 and 9
p["value_flag"] = np.select([value < 1_000, value > 100_000_000],
                            ["scaled_x1000", "scaled_div1000"], default="ok")
p["proposal_value_usd"] = np.select([value < 1_000, value > 100_000_000],
                                    [value * 1_000, value / 1_000], default=value)
print(p["value_flag"].value_counts())
print(f"Clean proposals: {len(p):,}")

Test records removed: 50
Dates that failed to parse: 0
Unmatched offices: 0
Unmatched practices: 0
value_flag
ok                23775
scaled_x1000        596
scaled_div1000        1
Name: count, dtype: int64
Clean proposals: 24,372


In [22]:
print("Offices:\n", p["office"].value_counts().sort_index().to_string(), "\n")
print("Practices:\n", p["practice"].value_counts().sort_index().to_string(), "\n")

Offices:
 office
Boston       1968
Chicago      1968
Dubai        1181
London       1894
Madrid       1827
Munich       1777
New Delhi    1222
New York     1987
Paris        1821
Riyadh       1101
Sao Paulo    1936
Singapore    1870
Sydney       1911
Tokyo        1909 

Practices:
 practice
Consumer Products                2063
Customer Strategy & Marketing    2091
Digital                          2309
Energy & Natural Resources       1584
Financial Services               2457
Healthcare                       2213
Industrials                      1815
M&A                              2240
Performance Improvement          1849
Strategy                         2983
Technology, Media & Telecom      2768 



## Clean events table

In [31]:
ev = events_raw.copy()
ev["proposal_id"] = ev["proposal_id"].str.strip().str.lstrip("0")
ev = ev[~ev["proposal_id"].isin(test_ids)]
ev["event_date"] = pd.to_datetime(ev["event_date"], format="%Y-%m-%d")
print("Events not matching a clean proposal:", (~ev["proposal_id"].isin(p["proposal_id"])).sum())

decisions = ev[ev["status"].isin(["Won", "Lost", "Withdrawn"])]
print("Proposals with more than one decision:", decisions["proposal_id"].duplicated().sum())

decision = decisions.set_index("proposal_id")[["status", "event_date"]] \
                    .rename(columns={"status": "outcome", "event_date": "decision_date"})
review = ev[ev["status"] == "Under Review"].groupby("proposal_id")["event_date"].min().rename("review_date")

fact = (p.merge(decision, left_on="proposal_id", right_index=True, how="left")
         .merge(review, left_on="proposal_id", right_index=True, how="left"))
fact["outcome"] = fact["outcome"].fillna("Open")                            # issue 10
fact["turnaround_calendar_days"] = (fact["decision_date"] - fact["submitted_date"]).dt.days
fact["submitted_month"] = fact["submitted_date"].dt.to_period("M").dt.to_timestamp()
fact["submitted_quarter"] = fact["submitted_date"].dt.to_period("Q").astype(str)

print(fact["outcome"].value_counts())
print("Negative turnaround:", (fact["turnaround_calendar_days"] < 0).sum())
fact.head()

Events not matching a clean proposal: 0
Proposals with more than one decision: 0
outcome
Lost         13169
Won           8987
Open          1152
Withdrawn     1064
Name: count, dtype: int64
Negative turnaround: 0


,proposal_id,submitted_date,office,client_name,practice,proposal_value_usd,value_flag,outcome,decision_date,review_date,turnaround_calendar_days,submitted_month,submitted_quarter
0,995,2025-12-24,Chicago,CL-4322,M&A,278900.0,ok,Lost,2026-01-16,2025-12-31,23.0,2025-12-01,2025Q4
1,14916,2026-03-20,Sydney,CL-6352,Performance Improvement,204000.0,ok,Won,2026-05-06,2026-03-27,47.0,2026-03-01,2026Q1
2,10088,2025-01-30,Tokyo,CL-6689,Financial Services,914100.0,ok,Lost,2025-04-08,2025-02-06,68.0,2025-01-01,2025Q1
3,5300,2025-01-13,Munich,CL-8209,Customer Strategy & Marketing,341800.0,ok,Won,2025-01-28,NaT,15.0,2025-01-01,2025Q1
4,1767,2025-08-07,Paris,CL-6153,Digital,206000.0,ok,Won,2025-09-03,2025-08-12,27.0,2025-08-01,2025Q3


## Task 1 : Get Public holidays 

In [26]:
import requests

def fetch_holidays(country_codes, years):
    rows, status = [], []
    for cc in country_codes:
        for year in years:
            url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{cc}"
            try:
                r = requests.get(url, timeout=20)
                data = r.json() if r.status_code == 200 and r.text.strip() else []
            except (requests.RequestException, ValueError) as err:
                print(f"  ! {cc} {year}: request failed ({err})")
                data = []
            national = [h for h in data if h.get("global")]
            status.append({"country_code": cc, "year": year, "holidays_found": len(national),
                           "holidays_available": len(national) > 0})
            rows += [{"country_code": cc, "date": h["date"], "name": h["name"],
                      "local_name": h["localName"]} for h in national]
    return pd.DataFrame(rows), pd.DataFrame(status)

holidays, holiday_status = fetch_holidays(sorted(offices["country_code"].unique()), [2025, 2026])
holidays.to_csv("C:/Users/rahulg/Downloads/PPK_Ops_Analytics_Case3/Outputs/public_holidays_2025_2026.csv", index=False)
holiday_status

,country_code,year,holidays_found,holidays_available
0,AE,2025,0,False
1,AE,2026,0,False
2,AU,2025,7,True
3,AU,2026,6,True
4,BR,2025,14,True
5,BR,2026,14,True
6,DE,2025,9,True
7,DE,2026,9,True
8,ES,2025,10,True
9,ES,2026,10,True


In [32]:
holiday_map = {cc: pd.to_datetime(grp["date"]).values.astype("datetime64[D]")
               for cc, grp in holidays.groupby("country_code")}
available = holiday_status.groupby("country_code")["holidays_available"].all()

# country code per proposal, used only for the calculation (not stored in fact)
office_cc = fact["office"].map(offices.set_index("office")["country_code"])

decided = fact["decision_date"].notna()
fact["turnaround_business_days"] = np.nan
for cc, idx in fact[decided].groupby(office_cc[decided]).groups.items():
    rows = fact.loc[idx]
    fact.loc[idx, "turnaround_business_days"] = np.busday_count(
        rows["submitted_date"].values.astype("datetime64[D]"),
        rows["decision_date"].values.astype("datetime64[D]"),
        holidays=holiday_map.get(cc, []))

# attribute of the office/country -> lives in the dimension
offices["holidays_available"] = offices["country_code"].map(available)

fact.loc[decided, ["office", "submitted_date", "decision_date",
                   "turnaround_calendar_days", "turnaround_business_days"]].head(10)

,office,submitted_date,decision_date,turnaround_calendar_days,turnaround_business_days
0,Chicago,2025-12-24,2026-01-16,23.0,15.0
1,Sydney,2026-03-20,2026-05-06,47.0,31.0
2,Tokyo,2025-01-30,2025-04-08,68.0,45.0
3,Munich,2025-01-13,2025-01-28,15.0,11.0
4,Paris,2025-08-07,2025-09-03,27.0,18.0
5,New York,2026-03-02,2026-04-08,37.0,27.0
6,Sydney,2025-12-09,2025-12-22,13.0,9.0
7,New York,2026-06-15,2026-07-17,32.0,22.0
8,Chicago,2025-08-20,2025-09-25,36.0,25.0
9,Munich,2025-09-30,2025-10-21,21.0,14.0


## Export data to csv

In [34]:
fact_cols = ["proposal_id", "submitted_date", "submitted_month", "submitted_quarter", "office",
             "practice", "client_name", "proposal_value_usd", "value_flag", "outcome",
             "review_date", "decision_date", "turnaround_calendar_days", "turnaround_business_days"]
fact[fact_cols].to_csv("C:/Users/rahulg/Downloads/PPK_Ops_Analytics_Case3/Outputs/fact_proposals.csv", index=False)
offices.to_csv("C:/Users/rahulg/Downloads/PPK_Ops_Analytics_Case3/Outputs/dim_office.csv", index=False)          # office, country, country_code, subdivision, region, holidays_available
ev.to_csv("C:/Users/rahulg/Downloads/PPK_Ops_Analytics_Case3/Outputs/fact_status_events.csv", index=False)       # optional, for status-funnel visuals